In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import time
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast

In [22]:
'''import cv2
import os
import glob

# 1. Path to the parent folder where your 'train' and 'validation' directories are.
BASE_DATA_DIR = './data' 

# 2. Desired output image size.
OUTPUT_SIZE = (72, 72)

def preprocess_images():
    """
    Crops and resizes all silhouette images in the source directories and saves
    them to new 'processed' directories.
    """
    print("Starting image preprocessing...")
    
    # Define source and destination directories
    source_dirs = {
        'train': os.path.join(BASE_DATA_DIR, 'train'),
        'validation': os.path.join(BASE_DATA_DIR, 'validation')
    }
    
    dest_dirs = {
        'train': os.path.join(BASE_DATA_DIR, 'train_processed'),
        'validation': os.path.join(BASE_DATA_DIR, 'validation_processed')
    }

    # Loop through 'train' and 'validation' sets
    for set_name, source_path in source_dirs.items():
        dest_path = dest_dirs[set_name]
        
        print(f"\nProcessing images in: '{source_path}'")
        print(f"Saving processed images to: '{dest_path}'")

        # Create the destination directory if it doesn't exist
        if not os.path.exists(dest_path):
            os.makedirs(dest_path)

        # Get all subject subdirectories (e.g., '001', '002')
        subject_dirs = sorted([d for d in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, d))])

        for subject_id in subject_dirs:
            print(f"  - Processing Subject: {subject_id}")
            subject_source_path = os.path.join(source_path, subject_id)
            subject_dest_path = os.path.join(dest_path, subject_id)

            # Create the subject's subdirectory in the destination
            if not os.path.exists(subject_dest_path):
                os.makedirs(subject_dest_path)

            # Find all .png images for the subject
            image_files = glob.glob(os.path.join(subject_source_path, '*.png'))

            for image_path in image_files:
                try:
                    # 1. Load the image
                    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                    
                    # 2. Find the bounding box of the silhouette
                    #    This creates a binary image (0 or 255) and finds contours.
                    _, thresh = cv2.threshold(img, 1, 255, cv2.THRESH_BINARY)
                    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    
                    if contours:
                        # Get the bounding box of the largest contour
                        cnt = max(contours, key=cv2.contourArea)
                        x, y, w, h = cv2.boundingRect(cnt)
                        
                        # 3. Crop the image to the bounding box
                        cropped_img = img[y:y+h, x:x+w]
                        
                        # 4. Resize the cropped image to the target size
                        resized_img = cv2.resize(cropped_img, OUTPUT_SIZE, interpolation=cv2.INTER_AREA)
                        
                        # 5. Save the processed image
                        file_name = os.path.basename(image_path)
                        save_path = os.path.join(subject_dest_path, file_name)
                        cv2.imwrite(save_path, resized_img)

                except Exception as e:
                    print(f"    - Could not process {image_path}. Error: {e}")

    print("\nImage preprocessing completed successfully!")'''




'import cv2\nimport os\nimport glob\n\n# 1. Path to the parent folder where your \'train\' and \'validation\' directories are.\nBASE_DATA_DIR = \'./data\' \n\n# 2. Desired output image size.\nOUTPUT_SIZE = (72, 72)\n\ndef preprocess_images():\n    """\n    Crops and resizes all silhouette images in the source directories and saves\n    them to new \'processed\' directories.\n    """\n    print("Starting image preprocessing...")\n    \n    # Define source and destination directories\n    source_dirs = {\n        \'train\': os.path.join(BASE_DATA_DIR, \'train\'),\n        \'validation\': os.path.join(BASE_DATA_DIR, \'validation\')\n    }\n    \n    dest_dirs = {\n        \'train\': os.path.join(BASE_DATA_DIR, \'train_processed\'),\n        \'validation\': os.path.join(BASE_DATA_DIR, \'validation_processed\')\n    }\n\n    # Loop through \'train\' and \'validation\' sets\n    for set_name, source_path in source_dirs.items():\n        dest_path = dest_dirs[set_name]\n        \n        prin

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import time
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast

# --- Configuration ---
DATA_DIR = '.' 
NUM_SUBJECTS_TO_TRAIN = 123

# --- Model & Training Parameters ---
NUM_CLASSES = NUM_SUBJECTS_TO_TRAIN if NUM_SUBJECTS_TO_TRAIN is not None else 123
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 0.001

# --- NEW: Definition of a Simple CNN Model ---
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        # Convolutional layers
        self.conv_layers = nn.Sequential(
            # Input: 3 x 72 x 72
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Output: 16 x 36 x 36

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Output: 32 x 18 x 18

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
            # Output: 64 x 9 x 9
        )
        # Fully connected layers (classifier)
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 9 * 9, 512), # 64 channels * 9x9 image size
            nn.ReLU(),
            nn.Linear(512, num_classes) # Output layer
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

def train_gait_model():
    """
    Main function to set up the dataset, build the model, and run the training loop.
    """
    print(f"Initializing OPTIMIZED training process for {NUM_CLASSES} subjects...")

    # --- Set up Data Transformations ---
    data_transforms = {
        'train': transforms.Compose([
            transforms.Resize((72, 72)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ]),
        'validation': transforms.Compose([
            transforms.Resize((72, 72)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ]),
    }

    # --- Load the Datasets and Create DataLoaders ---
    train_dir = os.path.join(DATA_DIR, 'train_processed')
    validation_dir = os.path.join(DATA_DIR, 'validation_processed')
    
    full_datasets = {
        'train': datasets.ImageFolder(train_dir, data_transforms['train']),
        'validation': datasets.ImageFolder(validation_dir, data_transforms['validation'])
    }
    
    if NUM_SUBJECTS_TO_TRAIN is not None:
        all_classes = sorted(full_datasets['train'].classes)
        subset_classes = all_classes[:NUM_SUBJECTS_TO_TRAIN]
        class_to_idx_subset = {cls_name: i for i, cls_name in enumerate(subset_classes)}
        
        for phase in ['train', 'validation']:
            full_dataset = full_datasets[phase]
            
            # Use only samples from the subset of classes
            new_samples = []
            for path, old_idx in full_dataset.samples:
                class_name = all_classes[old_idx]
                if class_name in class_to_idx_subset:
                     new_samples.append((path, class_to_idx_subset[class_name]))
            
            full_datasets[phase].samples = new_samples
            full_datasets[phase].classes = subset_classes
            full_datasets[phase].class_to_idx = class_to_idx_subset
            full_datasets[phase].targets = [s[1] for s in new_samples]

    dataloaders = {
        'train': DataLoader(full_datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True),
        'validation': DataLoader(full_datasets['validation'], batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)
    }

    dataset_sizes = {x: len(full_datasets[x]) for x in ['train', 'validation']}
    print(f"Training data size: {dataset_sizes['train']}")
    print(f"Validation data size: {dataset_sizes['validation']}")

    # --- CHANGED: Build the Simple CNN Model ---
    model = SimpleCNN(num_classes=NUM_CLASSES)
    print("Using SimpleCNN model for this training run.")

    # --- Set up for Training ---
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Training will use device: {device}")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    scaler = GradScaler()
    use_amp = torch.cuda.is_available()

    # --- Start the Training Loop ---
    start_time = time.time()
    print("\nStarting model training...")

    for epoch in range(NUM_EPOCHS):
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}')
        print('-' * 10)

        for phase in ['train', 'validation']:
            model.train() if phase == 'train' else model.eval()

            running_loss = 0.0
            running_corrects = 0

            progress_bar = tqdm(dataloaders[phase], desc=f'{phase.capitalize()} Phase')
            
            for inputs, labels in progress_bar:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad(set_to_none=True)

                with autocast(enabled=use_amp):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                
                _, preds = torch.max(outputs, 1)

                if phase == 'train':
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    time_elapsed = time.time() - start_time
    print(f"\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    
    # Save the trained model
    torch.save(model.state_dict(), f'gait_simple_cnn_model_{NUM_CLASSES}_subjects.pth')
    print(f"Model saved successfully to 'gait_simple_cnn_model_{NUM_CLASSES}_subjects.pth'")




In [24]:
import cv2
from pathlib import Path
import numpy as np


def preprocess_image(image_path, output_size=(72, 72)):

    try:
        # 1. Load image as grayscale
        img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        
        if img is None:
            raise ValueError(f"Could not load image: {image_path}")
        
        # 2. Find bounding box of silhouette
        _, thresh = cv2.threshold(img, 1, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            print(f"Warning: No silhouette found, returning resized original")
            return cv2.resize(img, output_size, interpolation=cv2.INTER_AREA)
        
        # 3. Crop to bounding box
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        cropped_img = img[y:y+h, x:x+w]
        
        # 4. Resize to 72x72
        resized_img = cv2.resize(cropped_img, output_size, interpolation=cv2.INTER_AREA)
        
        return resized_img
    
    except Exception as e:
        print(f"Error preprocessing {image_path}: {e}")
        return None

def load_model(model_path, num_classes=123):
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = SimpleCNN(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    print(f"✓ Model loaded from: {model_path}")
    print(f"✓ Running on: {device}")
    
    return model, device


def predict_subject(image_path, model, device):
    # Step 1: Preprocess image
    print(f"\n1. Preprocessing image: {image_path}")
    preprocessed = preprocess_image(image_path)
    
    if preprocessed is None:
        return {'error': 'Failed to preprocess image'}
    
    # Step 2: Convert to tensor
    print(f"2. Converting to tensor...")
    # Convert grayscale to RGB (3 channels)
    rgb_img = cv2.cvtColor(preprocessed, cv2.COLOR_GRAY2BGR)
    tensor = transforms.ToTensor()(rgb_img)  # (3, 72, 72)
    
    # Normalize with same values as training
    normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    tensor = normalize(tensor)
    
    # Add batch dimension
    tensor = tensor.unsqueeze(0)  # (1, 3, 72, 72)
    tensor = tensor.to(device)
    
    # Step 3: Inference
    print(f"3. Running inference...")
    with torch.no_grad():
        outputs = model(tensor)
    
    # Step 4: Get predictions
    print(f"4. Processing results...")
    softmax_scores = torch.nn.functional.softmax(outputs, dim=1)[0]
    confidence, predicted_class = torch.max(softmax_scores, dim=0)
    
    predicted_class = predicted_class.item()
    confidence = confidence.item()
    all_confidences = softmax_scores.cpu().numpy()
    
    return {
        'predicted_class': predicted_class,
        'confidence': confidence,
        'all_confidences': all_confidences
    }

def display_results(result, image_path=None):
    print("GAIT RECOGNITION PREDICTION")
    
    if image_path:
        print(f"Image: {Path(image_path).name}")
    
    if 'error' in result:
        print(f" Error: {result['error']}")
    else:
        print(f"\n✓ Predicted Subject: Subject_{result['predicted_class']:03d}")
        print(f"✓ Confidence: {result['confidence']:.4f} ({result['confidence']*100:.2f}%)")
        
        # Top 3 predictions
        top_3_idx = np.argsort(result['all_confidences'])[-3:][::-1]
        print(f"\nTop 3 Predictions:")
        for rank, idx in enumerate(top_3_idx, 1):
            print(f"  {rank}. Subject_{idx:03d}: {result['all_confidences'][idx]:.4f}")
    



In [25]:
from collections import Counter

def predict_subjects_in_folder_majority(folder_path, model, device):
    folder_path = Path(folder_path)
    image_paths = list(folder_path.glob("*.png")) + list(folder_path.glob("*.jpg")) + list(folder_path.glob("*.jpeg"))

    if not image_paths:
        print(f"No images found in folder {folder_path}")
        return None

    predicted_classes = []

    for img_path in image_paths:
        # Preprocess using your exact process
        preprocessed_img = preprocess_image(str(img_path))
        if preprocessed_img is None:
            print(f"Skipping image (preprocess failed): {img_path.name}")
            continue

        # Convert to RGB tensor
        rgb_img = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        tensor = transforms.ToTensor()(rgb_img)
        normalize = transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
        tensor = normalize(tensor)
        tensor = tensor.unsqueeze(0).to(device)

        # Forward pass
        with torch.no_grad():
            outputs = model(tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)[0]
        pred_class = torch.argmax(probs).item()
        predicted_classes.append(pred_class)

        print(f"Image {img_path.name} predicted as Subject_{pred_class:03d}")

    if not predicted_classes:
        print("No valid predictions.")
        return None

    # Count most frequent prediction
    counter = Counter(predicted_classes)
    most_common_subject, count = counter.most_common(1)[0]

    print(f"\nMost frequent predicted subject across folder: Subject_{most_common_subject:03d} (predicted {count} times)")
    return most_common_subject


In [32]:
import torch

def load_full_model(model_path, num_classes=123):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load the entire model object directly
    model = torch.load(model_path, map_location=device)
    
    model.to(device)
    model.eval()
    
    print(f"✓ Full Model loaded from: {model_path}")
    print(f"✓ Running on: {device}")
    
    return model, device

# Usage:
# model_path = r"D:\vit study\Machine Learning\Gait\Models\gait_xception_model_overall123_subjects.pth"
# model, device = load_full_model(model_path)

In [37]:
model, device = load_full_model(r"D:\vit study\Machine Learning\Gait\Models\gait_xception_model_overall123_subjects.pth", num_classes=123)
folder_path = r"D:\vit study\Machine Learning\Gait\CASIA - B\CASIA - B\GaitDatasetB-silh\GaitDatasetB-silh\GaitDatasetB-silh\016\nm-02\018"

most_common_subject = predict_subjects_in_folder_majority(folder_path, model, device)
print(f"Final subject prediction: Subject_{most_common_subject:03d}")




C:\Users\ankit\AppData\Local\Temp\ipykernel_19368\3830901963.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path, map_location=device)


✓ Full Model loaded from: D:\vit study\Machine Learning\Gait\Models\gait_xception_model_overall123_subjects.pth
✓ Running on: cuda
Image 016-nm-02-018-001.png predicted as Subject_014
Image 016-nm-02-018-002.png predicted as Subject_014
Image 016-nm-02-018-003.png predicted as Subject_014
Image 016-nm-02-018-004.png predicted as Subject_014
Image 016-nm-02-018-005.png predicted as Subject_014
Image 016-nm-02-018-006.png predicted as Subject_014
Image 016-nm-02-018-007.png predicted as Subject_014
Image 016-nm-02-018-008.png predicted as Subject_014
Image 016-nm-02-018-009.png predicted as Subject_014
Image 016-nm-02-018-010.png predicted as Subject_014
Image 016-nm-02-018-011.png predicted as Subject_014
Image 016-nm-02-018-012.png predicted as Subject_014
Image 016-nm-02-018-013.png predicted as Subject_014
Image 016-nm-02-018-014.png predicted as Subject_014
Image 016-nm-02-018-015.png predicted as Subject_014
Image 016-nm-02-018-016.png predicted as Subject_014
Image 016-nm-02-018-0